# Trace Acquisition — Unmasked AEAD (ChaCha20-Poly1305)

This notebook implements the trace-acquisition pipeline for the
**unmasked (standard)** ChaCha20-Poly1305 AEAD core on the CW305 (Artix-7)
FPGA target.  It serves as the **baseline** against which the masked
implementation is compared.

## Differences from the Masked Flow

| Aspect | Masked (`01_`) | Unmasked (this notebook) |
|---|---|---|
| Bitstream | `aead_masked` | `aead_unmasked` |
| Expected TVLA result | Pass ($|t| < 4.5$) | Fail (clear leakage) |
| ADC gain | 26 dB | 30 dB (stronger signal) |

All other setup steps (scope init, clock sync, ITF driver, R-F-F-R
capture) are identical.  See `01_capture_aead_masked.ipynb` for detailed
documentation.

---

## 1 — Scope Setup & FPGA Programming

In [ ]:
import sys, os, time
import chipwhisperer as cw

# Resolve repo root robustly (works regardless of kernel cwd)
_nb_dir = os.path.dirname(os.path.abspath("__file__"))
for _c in [os.path.join(_nb_dir, ".."), _nb_dir]:
    if os.path.isdir(os.path.join(os.path.abspath(_c), "src")):
        REPO_ROOT = os.path.abspath(_c); break
else:
    REPO_ROOT = os.path.abspath(".")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.config import BITSTREAMS, DEFINES_FILE, DATA_COMBINED

PLATFORM        = "CWLITE"
TARGET_PLATFORM = "CW305_100t"
ILA_DEBUG       = False

# *** KEY DIFFERENCE: unmasked bitstream ***
BITSTREAM = BITSTREAMS["aead_unmasked"]

try:
    scope = cw.scope()
except Exception:
    for obj_name in ("scope", "target"):
        if obj_name in globals():
            try: globals()[obj_name].dis()
            except Exception: pass
    time.sleep(0.5)
    scope = cw.scope()

scope.default_setup(verbose=False)
scope.adc.offset = 0
scope.adc.basic_mode = "high"
scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2  = "disabled"

platform = "cw305"
fpga_id  = "100t" if TARGET_PLATFORM == "CW305_100t" else "35t"

print(f"Programming FPGA: {BITSTREAM}")
target = cw.target(None, cw.targets.CW305,
                   bsfile=BITSTREAM, force=(not ILA_DEBUG),
                   platform=platform,
                   defines_files=[DEFINES_FILE])

## 2 — ADC & Clock Configuration

In [ ]:
scope.adc.samples = 10000
target.pll.pll_outfreq_set(15E6, 1)
target._clksleeptime = 150
scope.gain.db = 30  # Higher gain for unmasked (stronger leakage signal)

print("CLOCK SYNC: adc_src = extclk_x4")
scope.clock.adc_src = "extclk_x4"
scope.clock.freq_ctr_src = "extclk"
scope.adc.decimate = 1
scope.adc.offset = 0

target.vccint_set(1.0)
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True,  1)
target.pll.pll_outenable_set(False, 2)

scope.clock.reset_adc()
time.sleep(0.5)
assert scope.clock.adc_locked, "ADC failed to lock!"

project = cw.create_project("projects/AEAD_STD_CW305.cwp", overwrite=True)
print(f"ADC locked: {scope.clock.adc_locked}")

## 3 — Register Driver & Helpers

In [ ]:
from src.itf_driver import patch_target
from src.aead_helpers import (
    preload_constant_data, run_aead_test, run_aead_key_only,
    reference_chacha20_poly1305, get_last_trace_codes, aead_load_key,
)

patch_target(target)
target.itf_clk_div_set(2)
print("ITF driver patched.")

## 4 — Functional Verification

In [ ]:
ct_ref, tag_ref = reference_chacha20_poly1305()
ct_hw, tag_hw, cycles = run_aead_test(target)
assert ct_hw == ct_ref and tag_hw == tag_ref, "Mismatch!"
print(f"Functional check passed ({cycles} cycles).")

## 5 — Bulk TVLA Acquisition (R-F-F-R)

For the unmasked baseline, a smaller dataset (e.g. 10 000 traces) is
usually sufficient to demonstrate clear leakage.  Adjust `N_TRACES_TARGET`
as needed.

In [ ]:
import random, h5py, numpy as np
from tqdm.auto import tqdm

N_TRACES_TARGET = 10_000
WINDOW_SIZE     = scope.adc.samples
BATCH_SIZE      = 500
MAX_RETRIES     = 5

OUTPUT_FILE = os.path.join(
    DATA_COMBINED,
    "traces_combined_STD_unmasked_10k_smpl.h5"
)

KEY_FIXED = 0x808182838485868788898A8B8C8D8E8F909192939495969798999A9B9C9D9E9F

print("Pre-generating random keys...")
random_keys = [random.getrandbits(256) for _ in range(N_TRACES_TARGET)]

def capture_one(key_int):
    for attempt in range(MAX_RETRIES + 1):
        scope.arm()
        run_aead_key_only(target, key_int)
        if not scope.capture():
            tr = scope.get_last_trace()
            return (np.asarray(tr) * 65536).astype(np.int16)
        if attempt == MAX_RETRIES:
            return None
    return None

N_TRACES_TARGET -= N_TRACES_TARGET % 2
iterations = N_TRACES_TARGET // 2

if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

preload_constant_data(target)

try:
    with h5py.File(OUTPUT_FILE, "w") as f:
        dset_f = f.create_group("traces_fixed").create_dataset(
            "traces", (N_TRACES_TARGET, WINDOW_SIZE), dtype="int16",
            compression="lzf", chunks=(BATCH_SIZE, WINDOW_SIZE),
            maxshape=(None, WINDOW_SIZE))
        dset_r = f.create_group("traces_random").create_dataset(
            "traces", (N_TRACES_TARGET, WINDOW_SIZE), dtype="int16",
            compression="lzf", chunks=(BATCH_SIZE, WINDOW_SIZE),
            maxshape=(None, WINDOW_SIZE))

        buf_f = np.zeros((BATCH_SIZE, WINDOW_SIZE), dtype=np.int16)
        buf_r = np.zeros((BATCH_SIZE, WINDOW_SIZE), dtype=np.int16)
        buf_idx, global_idx, total_captured = 0, 0, 0

        try:
            for i in tqdm(range(iterations), desc="Acquiring (R-F-F-R)"):
                tr_r1 = capture_one(random_keys[i * 2])
                tr_f1 = capture_one(KEY_FIXED)
                tr_f2 = capture_one(KEY_FIXED)
                tr_r2 = capture_one(random_keys[i * 2 + 1])

                if any(t is None for t in (tr_r1, tr_f1, tr_f2, tr_r2)):
                    continue

                buf_r[buf_idx] = tr_r1;  buf_f[buf_idx] = tr_f1;  buf_idx += 1
                buf_f[buf_idx] = tr_f2;  buf_r[buf_idx] = tr_r2;  buf_idx += 1

                if buf_idx >= BATCH_SIZE:
                    s, e = global_idx, global_idx + buf_idx
                    dset_f[s:e] = buf_f[:buf_idx]
                    dset_r[s:e] = buf_r[:buf_idx]
                    global_idx += buf_idx
                    total_captured = global_idx
                    buf_idx = 0
                    f.flush()
        except KeyboardInterrupt:
            print("\nInterrupted — saving...")
        finally:
            if buf_idx > 0:
                s, e = global_idx, global_idx + buf_idx
                dset_f[s:e] = buf_f[:buf_idx]
                dset_r[s:e] = buf_r[:buf_idx]
                total_captured += buf_idx
            dset_f.resize((total_captured, WINDOW_SIZE))
            dset_r.resize((total_captured, WINDOW_SIZE))
            print(f"Saved {total_captured:,} traces.")
except Exception as e:
    print(f"Fatal error: {e}")

## 6 — Cleanup

In [ ]:
target.dis()
scope.dis()
print("Disconnected.")